# 🚀 Free Kaggle TPU v5e-8 Coding Host (Qwen2.5-Coder-32B / DeepSeek-R1-32B)

This notebook runs state-of-the-art open-weights coding models (**Qwen2.5-Coder-32B-Instruct** or **DeepSeek-R1-Distill-Qwen-32B**) **100% FREE** on Kaggle's **TPU VM v5e-8** (128GB HBM) with automatic fallback to **Dual NVIDIA Tesla T4** GPUs (30GB VRAM).

### ⚙️ Session Settings (Right Panel):
1. **Accelerator**: `TPU VM v5e-8` *(Fallback: `GPU T4 x2`)*
2. **Internet**: `Internet on` *(Requires phone-verified Kaggle account)*
3. **Persistence**: `Files only` (optional)
4. **No API tokens or credit card required!** Cloudflare Quick Tunnels provide instant public HTTPS URLs.

In [ ]:
%%bash
# CELL 1 — Detect Hardware & Install Serving Engine + Cloudflare Tunnel
set -e

echo "========================================="
echo "🔍 Detecting Hardware Environment..."
echo "========================================="

# 1. Download Cloudflare Quick Tunnel (100% Free, Zero Account Required)
if [ ! -f /usr/local/bin/cloudflared ]; then
    echo "⬇️ Downloading cloudflared binary..."
    curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
    chmod +x /usr/local/bin/cloudflared
fi
echo "✅ cloudflared ready: $(cloudflared --version)"

# 2. Detect Accelerator & Install vLLM Engine
if [ -n "$TPU_NAME" ] || [ -d "/sys/class/tpu" ] || [ -f "/lib/libtpu.so" ]; then
    echo "⚡ TPU Environment Detected (v5e-8)"
    pip install -q --upgrade pip
    pip install -q vllm-tpu -f https://storage.googleapis.com/libtpu-releases/index.html || pip install -q "vllm[tpu]"
    echo "ENV_TYPE=TPU" > /kaggle/working/env_profile.txt
elif command -v nvidia-smi &> /dev/null; then
    echo "🎮 NVIDIA GPU Environment Detected"
    nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
    pip install -q --upgrade pip
    pip install -q "vllm>=0.6.0"
    echo "ENV_TYPE=GPU" > /kaggle/working/env_profile.txt
else
    echo "⚠️ Neither TPU nor GPU detected! Make sure Settings -> Accelerator is set to TPU VM v5e-8 or GPU T4 x2."
    echo "ENV_TYPE=CPU" > /kaggle/working/env_profile.txt
fi

echo "========================================="
echo "✅ Dependencies & Serving Engine Installed!"
echo "========================================="


In [ ]:
# CELL 2 — Launch OpenAI-Compatible Server + Cloudflare Public Tunnel
import os, sys, time, subprocess, re, requests
from pathlib import Path

# Read environment type
env_type = "TPU"
if Path("/kaggle/working/env_profile.txt").exists():
    for line in Path("/kaggle/working/env_profile.txt").read_text().splitlines():
        if line.startswith("ENV_TYPE="):
            env_type = line.split("=")[1].strip()

print(f"[*] Active Hardware Mode: {env_type}")

# Model Selection:
# TPU v5e-8 has 128GB HBM memory -> runs full unquantized 32B model with 16k context effortlessly!
# GPU T4 x2 has 30GB total VRAM -> uses high-speed AWQ quantized 32B model across both GPUs.
if env_type == "TPU":
    MODEL_ID = "Qwen/Qwen2.5-Coder-32B-Instruct"  # Alternative: "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"
    MAX_LEN = "16384"
    EXTRA_FLAGS = [
        "--max-model-len", MAX_LEN,
    ]
else:  # Dual T4 GPU
    MODEL_ID = "Qwen/Qwen2.5-Coder-32B-Instruct-AWQ"
    MAX_LEN = "8192"
    EXTRA_FLAGS = [
        "--tensor-parallel-size", "2",
        "--max-model-len", MAX_LEN,
        "--gpu-memory-utilization", "0.92",
    ]

SERVED_NAME = "free-coder"
PORT = 8000
LOG_FILE = "/kaggle/working/vllm-server.log"
TUNNEL_LOG = "/kaggle/working/tunnel.log"

# 1. Clean up any previous server/tunnel instances
subprocess.run(["pkill", "-9", "-f", "vllm"], capture_output=True)
subprocess.run(["pkill", "-9", "-f", "cloudflared"], capture_output=True)
time.sleep(2)

# 2. Launch vLLM Server in Background
print(f"[*] Launching vLLM with {MODEL_ID} (Context: {MAX_LEN} tokens)...")
vllm_cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--served-model-name", SERVED_NAME,
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--trust-remote-code",
    *EXTRA_FLAGS
]

vllm_log = open(LOG_FILE, "w")
vllm_proc = subprocess.Popen(
    vllm_cmd,
    stdout=vllm_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
    stdin=subprocess.DEVNULL,
    close_fds=True,
)
print(f"[*] vLLM server process spawned (PID: {vllm_proc.pid})")

# 3. Launch Cloudflare Quick Tunnel in Background
print("[*] Starting Cloudflare Quick Tunnel (No auth token required)...")
tunnel_log = open(TUNNEL_LOG, "w")
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
    stdin=subprocess.DEVNULL,
    close_fds=True,
)

# 4. Extract Public Cloudflare URL from Tunnel Log
public_url = None
for _ in range(30):
    if Path(TUNNEL_LOG).exists():
        content = Path(TUNNEL_LOG).read_text()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if match:
            public_url = match.group(0)
            break
    time.sleep(1)

if not public_url:
    raise RuntimeError("Failed to extract Cloudflare public tunnel URL. Check /kaggle/working/tunnel.log")

api_base = f"{public_url}/v1"

# 5. Poll /v1/models until weights finish loading into TPU/GPU memory
print("[*] Waiting for model weights to load into memory (~60-120s)...", flush=True)
ready = False
for i in range(70):
    try:
        resp = requests.get(f"http://127.0.0.1:{PORT}/v1/models", timeout=3)
        if resp.status_code == 200:
            ready = True
            break
        print(f"    Loading model... ({i+1}/70) HTTP {resp.status_code}")
    except requests.RequestException:
        print(f"    Waiting for port {PORT}... ({i+1}/70)")
    time.sleep(5)

if not ready:
    print("❌ Timeout waiting for model to load. Inspecting recent log lines:")
    if Path(LOG_FILE).exists():
        print("\n".join(Path(LOG_FILE).read_text().splitlines()[-25:]))
    raise TimeoutError("Model failed to initialize in time.")

# 6. Print Formatted Connection Card for VS Code Extensions
print("\n" + "=" * 65)
print("🎉 FREE CLOUD TPU CODING ENDPOINT IS LIVE!")
print("=" * 65)
print(f"🌐 Base URL:   {api_base}")
print(f"🔑 API Key:    EMPTY  (or any dummy string)")
print(f"🧠 Model ID:   {SERVED_NAME}")
print(f"📏 Context:    {MAX_LEN} tokens")
print("=" * 65)
print("\n📋 Copy-Paste Config for VS Code (Roo Code / Cline / Continue):")
config_json = {
    "provider": "OpenAI Compatible",
    "baseUrl": api_base,
    "apiKey": "EMPTY",
    "modelId": SERVED_NAME,
    "contextWindow": int(MAX_LEN)
}
import json
print(json.dumps(config_json, indent=2))
print("=" * 65)


In [ ]:
# CELL 3 — Keep-Alive Heartbeat Daemon
from datetime import datetime
import time, requests

PORT = 8000
print('🚀 Serving active at:', api_base)
print('Started at:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Keep this cell running while coding in VS Code / Cursor.')
print('Press the Stop/Interrupt button when you are done to save quota.')

try:
    while True:
        time.sleep(300)
        try:
            check = requests.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=5)
            status = 'healthy' if check.ok else f'HTTP {check.status_code}'
        except Exception as exc:
            status = f'not reachable: {exc}'
        print(datetime.now().strftime('%H:%M:%S'), 'heartbeat:', status)
except KeyboardInterrupt:
    print('Keep-alive stopped. You may now stop the Kaggle session.')
